# Part 5 — Efficient Data Handling through Data Parallelism
**Project:** Financial Forecasting Frontier: Distributed ML  
**Contribution:** Individual — Suman  
**Domain:** Banking marketing analytics  
**Data:** Official AlmaBetter bank.csv, 4,521 client/contact records, 17 columns.  
**Objective:** Explain distributed data processing and estimate term-deposit subscription. This is an educational sample, not a credit eligibility or fraud system.  
The supplied EDA/ML templates guide the summary → business problem → data → transformations → analysis/model → conclusion structure. Part-specific questions below follow the official question documents.


## Q1. Data loading and partition strategy
Round-robin repartitioning balances rows across workers without using highly skewed job categories as partition keys. Aggregations still shuffle by key. Four partitions use two available executor cores over multiple tasks. Partitioning changes execution layout, not the statistical train/test split.

In [1]:
APP_NAME='Banking-Parallelism'
import sys
sys.path.insert(0, '/workspace')
from banking.common import *
prepare_dirs()
spark = session(APP_NAME)
df = load_bank(spark).repartition(4).cache()
print('Spark version:', spark.version)
print('Master:', spark.sparkContext.master)
print('Data partitions:', df.rdd.getNumPartitions())

df.show(5)
print('Rows per partition:',df.rdd.mapPartitions(lambda it:[sum(1 for _ in it)]).collect())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/21 04:54:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.3
Master: spark://spark-master:7077


Data partitions: 4


+---+-----------+--------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+---+
|age|        job| marital|education|default|balance|housing|loan| contact|day|month|duration|campaign|pdays|previous|poutcome|  y|
+---+-----------+--------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+---+
| 26|blue-collar| married|  primary|     no|   5116|    yes|  no|cellular| 22|  jul|     246|       3|   -1|       0| unknown| no|
| 38| management|  single| tertiary|     no|    427|    yes|  no|cellular|  6|  may|      17|       3|  169|       2| failure| no|
| 33| technician|  single| tertiary|     no|    696|     no| yes|cellular|  5|  feb|     193|       1|  176|       8| failure| no|
| 50| management| married| tertiary|     no|     78|    yes|  no|cellular| 14|  jan|     218|       3|  161|       9|   other| no|
| 44|     admin.|divorced|secondary|     no|    432|     no|  no|cellular|  9|  feb

Rows per partition: [1130, 1131, 1130, 1130]


## Q2. Parallel analysis: average balance and the unavailable loan amount
Compute average balance with a distributed groupBy. The dataset contains only a yes/no loan flag, so ranking loan amounts is impossible. The explicit substitute ranks the five age decades with the most personal-loan holders and also reports prevalence and group size. These counts must never be presented as currency.

In [2]:
df.groupBy('job').agg(F.avg('balance').alias('avg_balance')).orderBy('job').show()
loan_groups=(df.withColumn('age_decade',(F.floor(F.col('age')/10)*10).cast('int')).groupBy('age_decade')
 .agg(F.count('*').alias('clients'),F.sum((F.col('loan')=='yes').cast('int')).alias('loan_holders'))
 .withColumn('loan_pct',100*F.col('loan_holders')/F.col('clients')).orderBy(F.desc('loan_holders'),'age_decade'))
loan_groups.show(5)
save_json('parallel_loan_groups.json',{'limitation':'No loan amount is present; this ranks counts only.','results':rows(loan_groups.limit(5))})


+-------------+------------------+
|          job|       avg_balance|
+-------------+------------------+
|       admin.|  1226.73640167364|
|  blue-collar| 1085.161733615222|
| entrepreneur|          1645.125|
|    housemaid|2083.8035714285716|
|   management|1766.9287925696594|
|      retired| 2319.191304347826|
|self-employed|1392.4098360655737|
|     services|1103.9568345323742|
|      student|1543.8214285714287|
|   technician|     1330.99609375|
|   unemployed|       1089.421875|
|      unknown|1501.7105263157894|
+-------------+------------------+



+----------+-------+------------+------------------+
|age_decade|clients|loan_holders|          loan_pct|
+----------+-------+------------+------------------+
|        30|   1808|         271|14.988938053097344|
|        40|   1203|         184|15.295095594347465|
|        50|    854|         160| 18.73536299765808|
|        20|    478|          68|14.225941422594142|
|        60|    113|           8| 7.079646017699115|
+----------+-------+------------+------------------+
only showing top 5 rows



## Q3. Train on partitioned data
A Spark logistic model distributes computation over row partitions and combines gradient statistics. It is not a separate independent model per partition. Use the same stable held-out split as Part 3. Resource monitoring below covers this training and the parallel processing tasks.

In [3]:
from banking.monitor import ResourceMonitor
monitor=ResourceMonitor().start()
from banking.modeling import make_pipeline,stable_split,evaluate
from pyspark.ml.classification import LogisticRegression
train,test=stable_split(features(df))
trained=make_pipeline(LogisticRegression(maxIter=40,regParam=0.1)).fit(train.repartition(4))
parallel_metrics=evaluate(trained.transform(test))
print('Held-out metrics for partitioned logistic model:',parallel_metrics)
save_json('parallel_model.json',parallel_metrics)


26/10/21 04:56:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


26/10/21 04:57:08 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Held-out metrics for partitioned logistic model: {'accuracy': 0.8881278538812786, 'precision': 0.45454545454545453, 'recall': 0.05154639175257732, 'f1': 0.09259259259259259, 'confusion': {'tn': 773, 'fp': 6, 'fn': 92, 'tp': 5}, 'roc_auc': 0.7058481002607093, 'pr_auc': 0.2643724760947851}


## Q4. Partition benchmark and resource monitoring
Compare 1, 2 and 4 partitions over three repetitions on a cached synthetic expansion of the original data (50 repeated copies, 226,050 rows). Repetitions are for demonstrating execution only; they do not create new clients and never enter model evaluation. Materialize each layout before timing to separate repartition cost. Compare results for correctness. Tiny jobs can be slower with more partitions because scheduling and shuffle overhead dominate.

In [4]:
import time,statistics
expanded=df.crossJoin(spark.range(50).withColumnRenamed('id','replica')).cache()
print('Synthetic benchmark rows:',expanded.count())
results=[];reference=None
for partitions in [1,2,4]:
    table=expanded.repartition(partitions).cache();table.count()
    samples=[]
    for repeat in range(3):
        start=time.perf_counter()
        out=table.groupBy('job').agg(F.round(F.avg('balance'),6).alias('avg_balance')).orderBy('job').collect()
        samples.append(time.perf_counter()-start)
        current=[r.asDict() for r in out]
        if reference is None: reference=current
        assert current==reference, 'Partitioning changed the answer'
    results.append({'partitions':partitions,'seconds':samples,'median_seconds':statistics.median(samples)})
    table.unpersist()
expanded.unpersist()
for item in results:print(item)
save_json('parallel_benchmark.json',results)


Synthetic benchmark rows: 226050


{'partitions': 1, 'seconds': [1.658956795999984, 0.694336499999963, 0.5722502559999612], 'median_seconds': 0.694336499999963}
{'partitions': 2, 'seconds': [2.302231971999845, 1.7368574510001054, 1.5461548109999512], 'median_seconds': 1.7368574510001054}
{'partitions': 4, 'seconds': [2.385744036999995, 1.2354557399999067, 1.0999513089998345], 'median_seconds': 1.2354557399999067}


## Q5. Task management and scheduling
Two independent summaries run concurrently using Python threads. Each thread sets its own Spark job group and FAIR scheduling pool. Spark schedules tasks across executor slots. Futures propagate failures; both results must complete. This demonstrates concurrency without claiming twice the speed.

In [5]:
from concurrent.futures import ThreadPoolExecutor
def summary_task(name,column):
    sc=spark.sparkContext
    sc.setLocalProperty('spark.scheduler.pool',name)
    sc.setJobGroup(name,'Parallel '+name)
    try:return name,rows(df.groupBy(column).count().orderBy(column))
    finally:
        sc.setLocalProperty('spark.scheduler.pool',None)
        sc.setLocalProperty('spark.jobGroup.id',None)
with ThreadPoolExecutor(max_workers=2) as executor:
    futures=[executor.submit(summary_task,'education_summary','education'),executor.submit(summary_task,'contact_summary','contact')]
    concurrent=dict(f.result() for f in futures)
print(concurrent)
resource=monitor.stop()
print('Driver process monitor:',resource)
save_json('parallel_resources.json',resource)
save_json('parallel_tasks.json',concurrent)
df.unpersist();spark.stop()


26/10/21 04:58:28 WARN FairSchedulableBuilder: A job was submitted with scheduler pool education_summary, which has not been configured. This can happen when the file that pools are read from isn't set, or when that file doesn't contain education_summary. Created education_summary with default configuration (schedulingMode: FIFO, minShare: 0, weight: 1)
26/10/21 04:58:28 WARN FairSchedulableBuilder: A job was submitted with scheduler pool contact_summary, which has not been configured. This can happen when the file that pools are read from isn't set, or when that file doesn't contain contact_summary. Created contact_summary with default configuration (schedulingMode: FIFO, minShare: 0, weight: 1)


{'education_summary': [{'education': 'primary', 'count': 678}, {'education': 'secondary', 'count': 2306}, {'education': 'tertiary', 'count': 1350}, {'education': 'unknown', 'count': 187}], 'contact_summary': [{'contact': 'cellular', 'count': 2896}, {'contact': 'telephone', 'count': 301}, {'contact': 'unknown', 'count': 1324}]}
Driver process monitor: {'scope': 'driver Python and child processes; excludes remote executors', 'sample_interval_seconds': 1, 'samples': 130, 'peak_rss_bytes': 929767424, 'peak_cpu_percent': 247.6, 'series': [{'time': 1792558580.8484974, 'rss_bytes': 635490304, 'cpu_percent': 0.0}, {'time': 1792558581.917894, 'rss_bytes': 643878912, 'cpu_percent': 156.1}, {'time': 1792558582.9214594, 'rss_bytes': 646762496, 'cpu_percent': 181.3}, {'time': 1792558583.9228554, 'rss_bytes': 649121792, 'cpu_percent': 142.8}, {'time': 1792558584.9352965, 'rss_bytes': 654233600, 'cpu_percent': 215.4}, {'time': 1792558585.9441714, 'rss_bytes': 657072128, 'cpu_percent': 142.7}, {'time'

## Conclusion
Partitioning, caching, built-in SQL transformations and explicit scheduling make work manageable. Driver CPU/memory samples are not executor measurements; Docker container statistics are collected separately for cluster resource evidence. All worker containers share one physical laptop, so this verifies distributed processes and networking, not multi-machine fault tolerance or production speedup.